# Task 2 — Generate Product Descriptions

For each of the 50 products in the dataset, **Gemma-2-9b-it** (via Nebius Token Factory) generates a persuasive 50–90 word description. The generated text, latency, and token counts are collected and saved to `task_3/assignment_01.xlsx` with blank evaluation columns ready for Task 3.

In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import (
    DATASET_PATH,
    GENERATION_MODEL,
    NEBIUS_BASE_URL,
    OUTPUT_EXCEL,
)
from shared.constants import CRITERIA, SYSTEM_PROMPT
from shared.utils import build_user_message

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

In [2]:
df = pd.read_excel(DATASET_PATH)
print(f"{len(df)} products loaded")
df.head(3)

50 products loaded


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty


## System Prompt

The system prompt:
- Sets the role to e-commerce copywriter
- States the length constraint explicitly (50–90 words)
- Requires grounding — only use the provided product info
- Requests a persuasive, friendly tone

In [3]:
print(SYSTEM_PROMPT)
print("\n--- Example user message ---")
print(build_user_message(df.iloc[0]))

You are an e-commerce copywriter. Write a persuasive product description.

Rules:
- Length: exactly 50 to 90 words. Count carefully.
- Use ONLY the product information provided. Do not invent features, specs, or claims.
- Tone: friendly, confident, and professional. Write as if for a product listing page.
- Highlight key features and materials that matter to buyers.
- Mention the warranty naturally if it adds value.
- Do not include headings, bullet points, or markdown. Output only the description text.

--- Example user message ---
Product: Apple iPhone 15 Pro
Attributes: features: A17 Pro chip, 120 Hz ProMotion display, USB‑C fast charging; dimensions: compact
Material: titanium frame, Ceramic Shield glass
Warranty: 1‑year limited warranty


## Generate Descriptions

In [4]:
def generate_description(row: pd.Series) -> dict:
    """Call the model and return description + metrics."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(row)},
    ]

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=200,
    )
    elapsed_ms = (time.perf_counter() - start) * 1000

    usage = response.usage
    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": round(elapsed_ms, 1),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
    }

In [5]:
results = []

for idx, row in df.iterrows():
    result = generate_description(row)
    results.append(result)
    print(f"[{idx + 1}/{len(df)}] {row['product_name']}: "
          f"{result['latency_ms']:.0f}ms, "
          f"{len(result['generated_description'].split())} words")

print(f"\nDone. Generated {len(results)} descriptions.")

[1/50] Apple iPhone 15 Pro: 1455ms, 66 words
[2/50] Samsung Galaxy S24 Ultra: 1058ms, 75 words
[3/50] Google Pixel 8 Pro: 938ms, 71 words
[4/50] Sony WH‑1000XM5 Headphones: 958ms, 62 words
[5/50] Bose QuietComfort Ultra Earbuds: 828ms, 71 words
[6/50] Amazon Echo Dot (5th Gen): 2048ms, 69 words
[7/50] Dell XPS 13 9310 Laptop: 1332ms, 79 words
[8/50] Apple MacBook Air 13″ (M3): 921ms, 65 words
[9/50] Microsoft Surface Pro 10: 717ms, 55 words
[10/50] Garmin Forerunner 255: 944ms, 73 words
[11/50] Fitbit Charge 6: 898ms, 76 words
[12/50] GoPro HERO12 Black: 1023ms, 76 words
[13/50] DJI Mini 4 Pro Drone: 1616ms, 65 words
[14/50] Nintendo Switch OLED: 878ms, 63 words
[15/50] PlayStation 5 Slim: 1006ms, 76 words
[16/50] Xbox Series X: 893ms, 65 words
[17/50] Instant Pot Duo 6‑Quart: 1661ms, 73 words
[18/50] Keurig K‑Supreme Plus Smart: 804ms, 60 words
[19/50] Vitamix 5200 Blender: 1131ms, 79 words
[20/50] Dyson V15 Detect Vacuum: 1016ms, 80 words
[21/50] iRobot Roomba j7+: 820ms, 67 words
[2

## Build Output DataFrame

In [6]:
results_df = pd.DataFrame(results)
output_df = pd.concat([df, results_df], axis=1)

# Add blank evaluation columns
for criterion in CRITERIA:
    output_df[criterion] = ""
output_df["final_score"] = ""

print(f"Output shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")
output_df.head(3)

Output shape: (50, 16)
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'Fluency', 'Grammar', 'Tone', 'Length', 'Grounding', 'Latency', 'Cost', 'final_score']


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Experience the power and performance of the Ap...,1454.7,303,98,,,,,,,,
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the stunning 200MP c...,1058.3,303,109,,,,,,,,
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture stunning moments with the Google Pixel...,937.5,300,96,,,,,,,,


In [7]:
OUTPUT_EXCEL.parent.mkdir(parents=True, exist_ok=True)
output_df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Saved to {OUTPUT_EXCEL}")

Saved to /Users/nisim/Desktop/Nebuis Academy/Lesson-1-homeowrk/task_3/assignment_01.xlsx


## Quick Stats

In [8]:
output_df["word_count"] = output_df["generated_description"].apply(lambda x: len(x.split()))

print("Latency (ms):")
print(output_df["latency_ms"].describe().round(1))
print()
print("Word count:")
print(output_df["word_count"].describe().round(1))
print()
in_range = output_df["word_count"].between(50, 90).sum()
print(f"Descriptions in 50-90 word range: {in_range}/{len(output_df)}")

Latency (ms):
count      50.0
mean     1043.3
std       271.9
min       716.9
25%       894.0
50%       957.0
75%      1103.1
max      2047.9
Name: latency_ms, dtype: float64

Word count:
count    50.0
mean     69.5
std       7.5
min      54.0
25%      65.0
50%      70.5
75%      74.8
max      93.0
Name: word_count, dtype: float64

Descriptions in 50-90 word range: 49/50
